# Migration: drop gold_source_coverage.event_id (one-off, 2026-10-11)

Asana 1218420283884341. `event_id` has been NULL on every row since the table was designed at citation grain. The live table has no Delta column mapping (`delta.columnMapping.mode` unset), so `ALTER TABLE ... DROP COLUMN` is not available, and enabling it would upgrade the table protocol irreversibly. Instead the table is replaced with the new definition (same schema as `notebook_01_document_matching` Cell 4, minus `event_id`).

`CREATE OR REPLACE TABLE` keeps the table's identity, owner and Delta history, so the previous contents stay readable with `VERSION AS OF <version before the migration>`. It leaves the table **empty**: run `refresh_gold_source_coverage` immediately afterwards. Safe to re-run (it does nothing harmful if already migrated).


In [0]:
%sql
-- Informational: shows whether event_id is still there. Expect one row before the migration, none after.
SELECT column_name FROM workspace.information_schema.columns
WHERE table_schema = 'genealogy' AND table_name = 'gold_source_coverage' AND column_name = 'event_id';

In [0]:
%sql
CREATE OR REPLACE TABLE genealogy.gold_source_coverage (
  person_gedcom_id    STRING,
  source_title        STRING,
  source_category     STRING,
  citation_date       STRING,
  citation_text       STRING,
  citation_url        STRING,
  has_document        BOOLEAN,
  has_transcript      BOOLEAN,
  matched_file_ids    STRING,
  ancestry_record_id  STRING,
  coverage_status     STRING,
  refreshed_at        TIMESTAMP
)
USING DELTA
COMMENT 'Source coverage audit: which tree citations have downloaded documents/transcripts.';

In [0]:
%sql
DESCRIBE TABLE genealogy.gold_source_coverage;